# Tunisian License Plate Detection and Recognition
## Complete Solution for AI Tunisia Hack 2019

This notebook implements a complete pipeline for:
1. **License Plate Detection**: Locate license plates in vehicle images
2. **License Plate Recognition**: Recognize the 7 digits on each plate

### Setup Instructions:
1. Upload your data files to Google Drive
2. Mount your Google Drive
3. Run all cells sequentially

## Step 1: Mount Google Drive and Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU availability
!nvidia-smi

## Step 2: Install Required Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python-headless pandas tqdm scikit-learn matplotlib
!pip install pillow==9.5.0

## Step 3: Configure Data Paths

Update these paths to match where you uploaded your data in Google Drive.

In [ ]:
# CONFIGURATION - UPDATE THESE PATHS
DATA_DIR = '/content/drive/MyDrive/license_plate_data'  # Change to your data directory
WORKING_DIR = '/content/license_plate_solution'
MODEL_DIR = f'{WORKING_DIR}/models'
OUTPUT_DIR = f'{WORKING_DIR}/output'

# Create working directories
import os
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Working directory: {WORKING_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 4: Download and Extract Data (if needed)

In [ ]:
# List files in your data directory to verify uploads
!ls -la "{DATA_DIR}"

# Extract zip files if they're not already extracted
import zipfile

# Extract detection training images
detection_zip = f'{DATA_DIR}/license_plates_detection_train.zip'
if os.path.exists(detection_zip):
    print("Extracting detection training images...")
    with zipfile.ZipFile(detection_zip, 'r') as zip_ref:
        zip_ref.extractall(f'{DATA_DIR}/detection_images')
    print("Done!")

# Extract recognition training images
recognition_zip = f'{DATA_DIR}/license_plates_recognition_train.zip'
if os.path.exists(recognition_zip):
    print("Extracting recognition training images...")
    with zipfile.ZipFile(recognition_zip, 'r') as zip_ref:
        zip_ref.extractall(f'{DATA_DIR}/recognition_images')
    print("Done!")

# Extract test images
test_zip = f'{DATA_DIR}/test.zip'
if os.path.exists(test_zip):
    print("Extracting test images...")
    with zipfile.ZipFile(test_zip, 'r') as zip_ref:
        zip_ref.extractall(f'{DATA_DIR}/test_images')
    print("Done!")

## Step 5: Define the Detection Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LicensePlateDetector(nn.Module):
    """CNN for detecting license plate bounding boxes"""
    
    def __init__(self):
        super(LicensePlateDetector, self).__init__()
        
        # Feature extraction backbone
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        # Bounding box regression head (4 outputs: ymin, xmin, ymax, xmax)
        self.bbox_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 4)
        )
        
        # Objectness head (is there a license plate?)
        self.objectness_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
    
    def forward(self, x):
        features = self.features(x)
        bbox = self.bbox_head(features)
        objectness = self.objectness_head(features)
        return bbox, objectness

print("Detection model defined successfully!")

## Step 6: Define the Recognition Model

In [ ]:
class CRNN(nn.Module):
    """CRNN for license plate character recognition"""
    
    def __init__(self, num_classes=10, num_digits=7):
        super(CRNN, self).__init__()
        self.num_classes = num_classes
        self.num_digits = num_digits
        
        # CNN feature extractor
        self.cnn = nn.Sequential(
            # Input: 3x48x168 (height, width)
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 64x24x84
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 128x12x42
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),  # 256x6x42 (only reduce height)
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),  # 512x3x42
            
            nn.Conv2d(512, 512, kernel_size=(2, 3), padding=0),  # 512x1x40
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        # RNN sequence modeling
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        # Classification head for each of the 7 digit positions
        self.classifier = nn.Linear(512, num_classes * num_digits)  # 512 = 256 * 2 (bidirectional)
    
    def forward(self, x):
        # CNN feature extraction
        features = self.cnn(x)  # (batch, 512, 1, 40)
        
        # Remove singleton dimension and transpose for RNN
        features = features.squeeze(2)  # (batch, 512, 40)
        features = features.permute(0, 2, 1)  # (batch, 40, 512)
        
        # RNN sequence modeling
        rnn_out, _ = self.rnn(features)  # (batch, 40, 512)
        
        # Use the last output for classification
        last_output = rnn_out[:, -1, :]  # (batch, 512)
        
        # Classify all 7 digits at once
        logits = self.classifier(last_output)  # (batch, 70)
        logits = logits.view(-1, self.num_digits, self.num_classes)  # (batch, 7, 10)
        
        return logits

print("Recognition model defined successfully!")

## Step 7: Data Loading and Preprocessing

In [ ]:
import cv2
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A

# Install albumentations if not already installed
try:
    import albumentations
except ImportError:
    !pip install albumentations
    import albumentations

class DetectionDataset(Dataset):
    """Dataset for license plate detection"""
    
    def __init__(self, csv_path, image_dir, transform=None, img_size=224):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.img_size = img_size
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_name = row['filename'] if 'filename' in row else row['image_id']
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Get bounding box coordinates
        ymin = row['ymin']
        xmin = row['xmin']
        ymax = row['ymax']
        xmax = row['xmax']
        
        # Apply transformations
        if self.transform:
            transformed = self.transform(image=image, bboxes=[[ymin, xmin, ymax, xmax]])
            image = transformed['image']
            bbox = transformed['bboxes'][0]
            ymin, xmin, ymax, xmax = bbox
        
        # Resize image
        image = cv2.resize(image, (self.img_size, self.img_size))
        
        # Normalize image
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
        
        # Normalize bounding box to [0, 1]
        h_orig, w_orig = self.df.iloc[idx].get('height', image.shape[1]), self.df.iloc[idx].get('width', image.shape[2])
        bbox = np.array([ymin/h_orig, xmin/w_orig, ymax/h_orig, xmax/w_orig], dtype=np.float32)
        
        # Objectness label (1 since all images have license plates)
        objectness = np.array([1.0], dtype=np.float32)
        
        return torch.tensor(image), torch.tensor(bbox), torch.tensor(objectness), img_name


class RecognitionDataset(Dataset):
    """Dataset for license plate recognition"""
    
    def __init__(self, csv_path, image_dir, transform=None, img_height=48, img_width=168):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.img_height = img_height
        self.img_width = img_width
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_name = row['filename'] if 'filename' in row else row['image_id']
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Apply transformations
        if self.transform:
            image = self.transform(image=image)['image']
        
        # Resize to fixed size
        image = cv2.resize(image, (self.img_width, self.img_height))
        
        # Normalize image
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
        
        # Get labels (7 digits)
        label_str = str(row['label']).zfill(7)  # Ensure 7 digits with leading zeros
        labels = np.array([int(d) for d in label_str], dtype=np.int64)
        
        return torch.tensor(image), torch.tensor(labels), img_name


# Define augmentations
detection_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(blur_limit=3, p=0.2),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=[]))

recognition_transform = A.Compose([
    A.HorizontalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(blur_limit=3, p=0.2),
    A.MotionBlur(blur_limit=3, p=0.2),
])

print("Datasets and transforms defined successfully!")

## Step 8: Training Functions

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt

def train_detector(model, train_loader, val_loader, device, epochs=50, lr=0.001):
    """Train the license plate detector"""
    
    criterion_bbox = nn.MSELoss()
    criterion_obj = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for images, bboxes, objectness, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} (Train)'):
            images = images.to(device)
            bboxes = bboxes.to(device)
            objectness = objectness.to(device)
            
            optimizer.zero_grad()
            
            pred_bboxes, pred_objectness = model(images)
            
            loss_bbox = criterion_bbox(pred_bboxes, bboxes)
            loss_obj = criterion_obj(pred_objectness, objectness)
            loss = loss_bbox + loss_obj
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
        
        train_loss /= len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for images, bboxes, objectness, _ in val_loader:
                images = images.to(device)
                bboxes = bboxes.to(device)
                objectness = objectness.to(device)
                
                pred_bboxes, pred_objectness = model(images)
                
                loss_bbox = criterion_bbox(pred_bboxes, bboxes)
                loss_obj = criterion_obj(pred_objectness, objectness)
                loss = loss_bbox + loss_obj
                
                val_loss += loss.item() * images.size(0)
        
        val_loss /= len(val_loader.dataset)
        history['val_loss'].append(val_loss)
        
        scheduler.step(val_loss)
        
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'{MODEL_DIR}/best_detector.pth')
            print(f'  -> Saved best model with val loss: {val_loss:.4f}')
    
    # Plot training history
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Detector Training History')
    plt.savefig(f'{MODEL_DIR}/detector_training_history.png')
    plt.show()
    
    return history


def train_recognizer(model, train_loader, val_loader, device, epochs=50, lr=0.001):
    """Train the license plate recognizer"""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} (Train)'):
            images = images.to(device)
            labels = labels.to(device)  # (batch, 7)
            
            optimizer.zero_grad()
            
            logits = model(images)  # (batch, 7, 10)
            
            # Reshape for loss calculation
            batch_size = logits.size(0)
            logits_flat = logits.view(-1, 10)  # (batch*7, 10)
            labels_flat = labels.view(-1)  # (batch*7)
            
            loss = criterion(logits_flat, labels_flat)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_size
            
            # Calculate accuracy
            preds = logits.argmax(dim=2)  # (batch, 7)
            correct += (preds == labels).sum().item()
            total += labels.numel()
        
        train_loss /= len(train_loader.dataset)
        train_acc = correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels, _ in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                
                logits = model(images)
                
                batch_size = logits.size(0)
                logits_flat = logits.view(-1, 10)
                labels_flat = labels.view(-1)
                
                loss = criterion(logits_flat, labels_flat)
                
                val_loss += loss.item() * batch_size
                
                preds = logits.argmax(dim=2)
                correct += (preds == labels).sum().item()
                total += labels.numel()
        
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_loss)
        
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'{MODEL_DIR}/best_recognizer.pth')
            print(f'  -> Saved best model with val loss: {val_loss:.4f}')
    
    # Plot training history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title('Recognizer Loss History')
    
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.set_title('Recognizer Accuracy History')
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/recognizer_training_history.png')
    plt.show()
    
    return history

print("Training functions defined successfully!")

## Step 9: Inference and Submission Generation

In [ ]:
def detect_license_plate(detector, image_path, device, img_size=224, conf_threshold=0.5):
    """Detect license plate in an image and return cropped plate"""
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = image.shape[:2]
    
    # Resize for detection
    image_resized = cv2.resize(image_rgb, (img_size, img_size))
    image_normalized = image_resized.astype(np.float32) / 255.0
    image_tensor = torch.tensor(np.transpose(image_normalized, (2, 0, 1))).unsqueeze(0).to(device)
    
    # Detect
    detector.eval()
    with torch.no_grad():
        bbox_pred, obj_pred = detector(image_tensor)
        
        obj_score = torch.sigmoid(obj_pred).item()
        
        if obj_score < conf_threshold:
            print(f"Warning: Low confidence ({obj_score:.2f}) for {image_path}")
            # Return center crop as fallback
            return image_rgb[h_orig//4:h_orig//4+h_orig//2, w_orig//4:w_orig//4+w_orig//2]
        
        bbox = bbox_pred.cpu().numpy()[0]
        
        # Convert normalized coordinates to original image coordinates
        ymin, xmin, ymax, xmax = bbox
        ymin, xmin = int(ymin * h_orig), int(xmin * w_orig)
        ymax, xmax = int(ymax * h_orig), int(xmax * w_orig)
        
        # Clip to image boundaries
        ymin = max(0, min(ymin, h_orig))
        xmin = max(0, min(xmin, w_orig))
        ymax = max(0, min(ymax, h_orig))
        xmax = max(0, min(xmax, w_orig))
        
        # Crop license plate
        plate_crop = image_rgb[ymin:ymax, xmin:xmax]
        
        return plate_crop


def recognize_plate(recognizer, plate_image, device, img_height=48, img_width=168):
    """Recognize characters in a license plate image"""
    
    # Preprocess plate image
    plate_resized = cv2.resize(plate_image, (img_width, img_height))
    plate_normalized = plate_resized.astype(np.float32) / 255.0
    plate_tensor = torch.tensor(np.transpose(plate_normalized, (2, 0, 1))).unsqueeze(0).to(device)
    
    # Recognize
    recognizer.eval()
    with torch.no_grad():
        logits = recognizer(plate_tensor)  # (1, 7, 10)
        preds = logits.argmax(dim=2)  # (1, 7)
        
        return preds[0].cpu().numpy()


def generate_submission(test_dir, detector, recognizer, device, output_file):
    """Generate submission file for test images"""
    
    # Get all test images
    test_images = sorted([f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    
    submission_rows = []
    
    for img_name in tqdm(test_images, desc='Generating submission'):
        img_path = os.path.join(test_dir, img_name)
        img_base = os.path.splitext(img_name)[0]
        
        # Detect license plate
        plate_crop = detect_license_plate(detector, img_path, device)
        
        # Recognize characters
        digits = recognize_plate(recognizer, plate_crop, device)
        
        # Generate one-hot encoded rows
        for i, digit in enumerate(digits):
            row_id = f"{img_base}_{i+1}"
            one_hot = [0] * 10
            one_hot[digit] = 1
            submission_rows.append([row_id] + one_hot)
    
    # Create DataFrame
    columns = ['id'] + [str(i) for i in range(10)]
    submission_df = pd.DataFrame(submission_rows, columns=columns)
    
    # Save to CSV
    submission_df.to_csv(output_file, index=False)
    print(f"Submission saved to: {output_file}")
    print(f"Total rows: {len(submission_df)}")
    
    return submission_df

print("Inference functions defined successfully!")

## Step 10: Run Training (Optional)

In [ ]:
# Configuration
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 0.001

# Check if data files exist
detection_csv = f'{DATA_DIR}/license_plates_detection_train.csv'
recognition_csv = f'{DATA_DIR}/license_plates_recognition_train.csv'
detection_images = f'{DATA_DIR}/detection_images'
recognition_images = f'{DATA_DIR}/recognition_images'

if not os.path.exists(detection_csv):
    print(f"Error: Detection CSV not found at {detection_csv}")
    print("Please upload the data files to your Google Drive and update DATA_DIR")
else:
    print("Starting training process...")
    
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # ========== TRAIN DETECTOR ==========
    print("\n" + "="*50)
    print("TRAINING LICENSE PLATE DETECTOR")
    print("="*50)
    
    # Create dataset and dataloader
    detection_dataset = DetectionDataset(detection_csv, detection_images, transform=detection_transform)
    
    # Split into train/val
    from torch.utils.data import random_split
    train_size = int(0.9 * len(detection_dataset))
    val_size = len(detection_dataset) - train_size
    train_dataset, val_dataset = random_split(detection_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Initialize model
    detector = LicensePlateDetector().to(device)
    
    # Train
    detector_history = train_detector(
        detector, train_loader, val_loader, device,
        epochs=NUM_EPOCHS, lr=LEARNING_RATE
    )
    
    # ========== TRAIN RECOGNIZER ==========
    print("\n" + "="*50)
    print("TRAINING LICENSE PLATE RECOGNIZER")
    print("="*50)
    
    # Create dataset and dataloader
    recognition_dataset = RecognitionDataset(recognition_csv, recognition_images, transform=recognition_transform)
    
    # Split into train/val
    train_size = int(0.9 * len(recognition_dataset))
    val_size = len(recognition_dataset) - train_size
    train_dataset, val_dataset = random_split(recognition_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Initialize model
    recognizer = CRNN(num_classes=10, num_digits=7).to(device)
    
    # Train
    recognizer_history = train_recognizer(
        recognizer, train_loader, val_loader, device,
        epochs=NUM_EPOCHS, lr=LEARNING_RATE
    )
    
    print("\n" + "="*50)
    print("TRAINING COMPLETE!")
    print("="*50)
    print(f"Best detector model saved to: {MODEL_DIR}/best_detector.pth")
    print(f"Best recognizer model saved to: {MODEL_DIR}/best_recognizer.pth")

## Step 11: Generate Submission File

In [ ]:
# Load best models and generate submission
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize models
detector = LicensePlateDetector().to(device)
recognizer = CRNN(num_classes=10, num_digits=7).to(device)

# Load trained weights
detector.load_state_dict(torch.load(f'{MODEL_DIR}/best_detector.pth', map_location=device))
recognizer.load_state_dict(torch.load(f'{MODEL_DIR}/best_recognizer.pth', map_location=device))

print("Models loaded successfully!")

# Generate submission
test_images_dir = f'{DATA_DIR}/test_images'
submission_file = f'{OUTPUT_DIR}/submission.csv'

if os.path.exists(test_images_dir):
    submission_df = generate_submission(
        test_images_dir, detector, recognizer, device, submission_file
    )
    
    # Display first few rows
    print("\nFirst 10 rows of submission:")
    display(submission_df.head(10))
    
    # Download submission file
    from google.colab import files
    files.download(submission_file)
    print("\nSubmission file downloaded!")
else:
    print(f"Test images directory not found: {test_images_dir}")
    print("Please upload test.zip to your Google Drive and extract it.")

## Step 12: Quick Test with Pre-trained Models (Alternative)

In [ ]:
# If you want to skip training and use pre-trained models,
# upload your trained models to Google Drive and run this cell:

USE_PRETRAINED = False  # Set to True if you have pre-trained models

if USE_PRETRAINED:
    # Update these paths to your pre-trained models
    PRETRAINED_DETECTOR = '/content/drive/MyDrive/pretrained_models/best_detector.pth'
    PRETRAINED_RECOGNIZER = '/content/drive/MyDrive/pretrained_models/best_recognizer.pth'
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize models
    detector = LicensePlateDetector().to(device)
    recognizer = CRNN(num_classes=10, num_digits=7).to(device)
    
    # Load pre-trained weights
    detector.load_state_dict(torch.load(PRETRAINED_DETECTOR, map_location=device))
    recognizer.load_state_dict(torch.load(PRETRAINED_RECOGNIZER, map_location=device))
    
    print("Pre-trained models loaded!")
    
    # Generate submission
    test_images_dir = f'{DATA_DIR}/test_images'
    submission_file = f'{OUTPUT_DIR}/submission_pretrained.csv'
    
    submission_df = generate_submission(
        test_images_dir, detector, recognizer, device, submission_file
    )
    
    # Download
    from google.colab import files
    files.download(submission_file)

## Troubleshooting Tips

1. **Out of Memory Error**: Reduce `BATCH_SIZE` to 16 or 8
2. **Slow Training**: Make sure you're using GPU (Runtime > Change runtime type > GPU)
3. **Poor Accuracy**: Increase `NUM_EPOCHS` or adjust learning rate
4. **Missing Files**: Verify all data files are uploaded to the correct Google Drive folder
5. **Import Errors**: Re-run the dependency installation cell

## Expected Results

- Detection model should achieve ~90%+ accuracy in locating plates
- Recognition model should achieve ~85%+ character accuracy
- Final submission will have 7 × N rows (where N = number of test images)

Good luck with the competition! 🇹🇳